# Egress Index, step 1 of 3: scoring the intersectionsWhich intersections in Santa Rosa have genuinely independent ways out, andhow many blocks sit trapped behind each single point of failure.**In:** OpenStreetMap road network, pulled live via OSMnx. No other inputs.**Out:** `egress_santa_rosa.geojson`, one point per intersection carryingscore, pocket size, and the street doing the trapping.**Runtime:** about 50 minutes on a free Colab CPU. Most of it is two loops,one max-flow per intersection and one min-cut per trapped spot.Part of https://github.com/jerrod-lessel/egress-indexAn earlier version of this notebook clipped the road graph at the cityboundary, which silently stranded any neighbourhood whose only way outcrossed the line. Those cells are gone. What survives is the rebuild thatfixed it, plus the checks that proved the fix was real.

## 1. Install the road-fetching tool

We need one outside helper, OSMnx, which pulls road networks from
OpenStreetMap (the free worldwide map database). Colab starts fresh each
session, so install it once at the top. Run again only if the notebook
later forgets it.

In [ ]:
# Install OSMnx, our road-fetching tool. "!" = a setup command, "-q" = quiet.
!pip install osmnx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.4 MB/s eta 0:00:00


## 2. Build the halo graphClipping the road network at the city boundary is wrong, and wrong in a waythat looks fine. A neighbourhood whose only way out runs through the countybefore coming back reads as trapped, because the road it uses was cut offthe edge of the data.So the graph is built on the city plus a 1,500 m halo, with boundary holesfilled in. Compute wide, show narrow: the halo exists so the math can seeroads leaving town, not so the map can draw ranch driveways.This also builds the flow graph. Every road gets capacity 1, so a max flowof 3 means three routes that share no pavement. That is the whole idea. Acorner touched by six roads that all feed the same one street out scores 1,which is the number that matters and the one you cannot see from thesidewalk.

In [ ]:
import osmnx as ox, networkx as nx, geopandas as gpdfrom shapely.geometry import Point, Polygon, MultiPolygonfrom shapely.prepared import prepplace    = "Santa Rosa, California, USA"HALO_M   = 1500SINK     = "SAFE_SINK"SAFE_ROAD_TYPES = {    "motorway", "motorway_link", "trunk", "trunk_link",    "primary", "primary_link", "secondary", "secondary_link",}# --- 1. City boundary, holes filled, buffered outward.def fill_holes(geom):    if geom.geom_type == "Polygon":        return Polygon(geom.exterior)    return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])city_poly = ox.geocode_to_gdf(place).geometry.iloc[0]solid = fill_holes(city_poly)halo  = (gpd.GeoSeries([solid], crs=4326)           .to_crs(3310).buffer(HALO_M).to_crs(4326).iloc[0])G_halo = ox.graph_from_polygon(halo, network_type="drive")print(f"G_halo: {len(G_halo.nodes):,} nodes  {len(G_halo.edges):,} edges  (expect 6,750 / 16,597)")# --- 2. Safety roads and the sink.safe_nodes_h = set()for u, v, data in G_halo.edges(data=True):    hwy = data.get("highway")    if isinstance(hwy, str):        hwy = [hwy]    if hwy and any(h in SAFE_ROAD_TYPES for h in hwy):        safe_nodes_h.add(u); safe_nodes_h.add(v)flowG_h = nx.DiGraph()for u, v, data in G_halo.edges(data=True):    flowG_h.add_edge(u, v, capacity=1)for n in safe_nodes_h:    flowG_h.add_edge(n, SINK, capacity=999999)print(f"safety nodes: {len(safe_nodes_h):,}  (expect 1,039)")# --- 3. What we display: inside the city, islands included, halo excluded.inside_test = prep(solid)display_nodes = {n for n, d in G_halo.nodes(data=True)                 if inside_test.contains(Point(d["x"], d["y"]))}# --- 4. Drop anything that cannot reach safety at all.stranded = set(flowG_h.nodes) - nx.ancestors(flowG_h, SINK) - {SINK}flowG_h.remove_nodes_from(stranded)display_nodes -= strandedprint(f"stranded dropped: {len(stranded)}  (expect 9)")print(f"display nodes: {len(display_nodes):,}  (expect 5,719)")print("\nready for the speed test ✅")

G_halo: 6,749 nodes  16,595 edges  (expect 6,750 / 16,597)
safety nodes: 1,038  (expect 1,039)
stranded dropped: 9  (expect 9)
display nodes: 5,718  (expect 5,719)

ready for cell 22 ✅


## 3. Speed test the cutoff shortcutEvery score is capped at 3, so computing a true max flow of 47 for adowntown intersection wastes 44 units of effort. NetworkX can stop countingonce it hits a target, using `cutoff=3`.Trust but verify. This scores 150 intersections the slow way and the fastway, checks that every answer matches, and reports the speedup. If theyagree, the fast version gets used for the full run. If they do not, itdoes not.

In [ ]:
import time, randomfrom networkx.algorithms.flow import shortest_augmenting_pathrandom.seed(1)todo = [n for n in display_nodes if n not in safe_nodes_h and n in flowG_h]test = random.sample(todo, 150)# --- The slow way: compute the true max flow, then cap it.t0 = time.time()slow = {n: min(int(nx.maximum_flow_value(flowG_h, n, SINK)), 3) for n in test}t_slow = time.time() - t0# --- The fast way: stop counting at 3, since that's all we ever wanted.#     Every road has capacity 1, so each path found is basically one BFS.#     cutoff=3 means it runs at most three of them, then quits.t0 = time.time()fast = {n: min(int(nx.maximum_flow_value(    flowG_h, n, SINK, flow_func=shortest_augmenting_path, cutoff=3)), 3) for n in test}t_fast = time.time() - t0# --- Do they agree? This is the part that matters.bad = [n for n in test if slow[n] != fast[n]]print(f"slow: {t_slow:.1f}s   fast: {t_fast:.1f}s   speedup: {t_slow/t_fast:.1f}x")print(f"disagreements: {len(bad)} / {len(test)}")for n in bad[:5]:    print(f"  node {n}: slow says {slow[n]}, fast says {fast[n]}")if not bad:    print(f"\n✅ safe. full run estimate: {t_fast/len(test)*len(todo)/60:.0f} min")else:    print("\n❌ do not use the shortcut. run the full scoring without the cutoff.")

slow: 66.2s   fast: 52.8s   speedup: 1.3x
disagreements: 0 / 150

✅ safe. full run estimate: 29 min


## 4. Score the whole cityOne max flow per intersection, against a virtual sink wired to everysafety road (motorway, trunk, primary, secondary). Intersections alreadysitting on a safety road are skipped, since they are the destination.Only the intersections meant for display get scored, which is Santa Rosaplus its county islands. The halo nodes are there to be routed through,not drawn.Roughly half an hour. Bigger graph than the clipped version, same idea.

In [ ]:
import timefrom networkx.algorithms.flow import shortest_augmenting_pathscores_h = {}todo = [n for n in display_nodes if n not in safe_nodes_h and n in flowG_h]print(f"scoring {len(todo)} intersections (skipping {len(display_nodes)-len(todo)} "      f"already on safety roads)\n")t0 = time.time()for i, n in enumerate(todo):    # cutoff=3 stops once it finds 3 ways out, which is all we ever asked for.    # Verified against the slow version in step 3.    scores_h[n] = min(int(nx.maximum_flow_value(        flowG_h, n, SINK, flow_func=shortest_augmenting_path, cutoff=3)), 3)    if i and i % 500 == 0:        rate = i / (time.time() - t0)        print(f"  {i:>5} / {len(todo)}   ~{(len(todo)-i)/rate/60:.1f} min left")print(f"\nDone in {(time.time()-t0)/60:.1f} min 🎯\n")# --- Side by side with the old, broken run.new_one   = sum(1 for s in scores_h.values() if s <= 1)new_two   = sum(1 for s in scores_h.values() if s == 2)new_three = sum(1 for s in scores_h.values() if s >= 3)new_safe  = len(display_nodes & safe_nodes_h)print(f"{'':<16}{'clipped':>10}{'halo':>10}{'change':>10}")print(f"{'-'*46}")for label, old, new in [    ("one way out",   1941, new_one),    ("two ways out",  1355, new_two),    ("three or more", 1196, new_three),    ("on a safety rd", 808,  new_safe),]:    print(f"{label:<16}{old:>10,}{new:>10,}{new-old:>+10,}")print(f"{'-'*46}")print(f"{'total shown':<16}{5302:>10,}{len(display_nodes):>10,}"      f"{len(display_nodes)-5302:>+10,}")# --- The interesting subset: looks like options, isn't.illusions = sum(1 for n in scores_h                if scores_h[n] <= 1 and G_halo.out_degree(n) >= 3)tips = sum(1 for n in scores_h           if scores_h[n] <= 1 and G_halo.out_degree(n) == 1)print(f"\nof the one-way-out spots:")print(f"  {tips:>5} are plain dead end tips (boring, correct)")print(f"  {illusions:>5} have 3+ roads leaving and still score 1  <- the point")

scoring 4906 intersections (skipping 812 already on safety roads)

    500 / 4906   ~26.0 min left
   1000 / 4906   ~22.9 min left
   1500 / 4906   ~19.9 min left
   2000 / 4906   ~17.0 min left
   2500 / 4906   ~14.1 min left
   3000 / 4906   ~11.2 min left
   3500 / 4906   ~8.3 min left
   4000 / 4906   ~5.4 min left
   4500 / 4906   ~2.4 min left

Done in 29.0 min 🎯

                   clipped      halo    change
----------------------------------------------
one way out          1,941     2,060      +119
two ways out         1,355     1,475      +120
three or more        1,196     1,371      +175
on a safety rd         808       812        +4
----------------------------------------------
total shown          5,302     5,718      +416

of the one-way-out spots:
   1513 are plain dead end tips (boring, correct)
    495 have 3+ roads leaving and still score 1  <- the point


## 5. Check the dead end count against the old graphMoving to the halo graph pushed the dead end count from 966 to 1,513,which is more than the 417 intersections it added. Spots with exactly twoways out fell from 497 to 52. Neither number changes any score, but bothwould end up in a writeup, so they get checked rather than picked.This rebuilds the old clipped graph and counts road degrees on both. Noscoring, so it is quick. If the old graph reproduces 966, the differenceis real and explainable. If it does not, the earlier number was stale.This cell is a diagnostic, not part of the pipeline. Nothing downstreamdepends on it.

In [ ]:
from collections import Counter

# --- 1. Rebuild the old clipped graph. Fetch only, no scoring.
G_old = ox.graph_from_place(place, network_type="drive")
print(f"G_old : {len(G_old.nodes):,} nodes  {len(G_old.edges):,} edges  (was 5,302 / 13,202)")
print(f"G_halo: {len(G_halo.nodes):,} nodes  {len(G_halo.edges):,} edges\n")

# --- 2. Old safety nodes, so we compare like for like.
safe_old = set()
for u, v, data in G_old.edges(data=True):
    hwy = data.get("highway")
    if isinstance(hwy, str):
        hwy = [hwy]
    if hwy and any(h in SAFE_ROAD_TYPES for h in hwy):
        safe_old.add(u); safe_old.add(v)

# --- 3. Every non-safety node scoring 1 has out_degree 1, by definition:
#        one road out means one path out. So we can count tips WITHOUT
#        any scoring at all, on both graphs, and compare directly.
old_pop = [n for n in G_old.nodes if n not in safe_old]
new_pop = [n for n in display_nodes if n not in safe_nodes_h]

d_old = Counter(G_old.out_degree(n) for n in old_pop)
d_new = Counter(G_halo.out_degree(n) for n in new_pop)

print(f"{'roads out':<12}{'clipped':>10}{'halo':>10}")
print("-" * 32)
for k in sorted(set(d_old) | set(d_new)):
    print(f"{k:<12}{d_old.get(k,0):>10,}{d_new.get(k,0):>10,}")
print("-" * 32)
print(f"{'total':<12}{len(old_pop):>10,}{len(new_pop):>10,}\n")

print(f"tips (out_degree 1): clipped {d_old[1]:,}  vs  halo {d_new[1]:,}")
print(f"  handoff claimed 966 for the clipped graph")
print(f"  -> {'reproduces' if abs(d_old[1]-966) < 30 else 'DOES NOT reproduce'}")

G_old : 5,301 nodes  13,200 edges  (was 5,302 / 13,202)
G_halo: 6,749 nodes  16,595 edges

roads out      clipped      halo
--------------------------------
0                    2         0
1                1,372     1,513
2                  194       145
3                2,407     2,686
4                  519       562
--------------------------------
total            4,494     4,906

tips (out_degree 1): clipped 1,372  vs  halo 1,513
  handoff claimed 966 for the clipped graph
  -> DOES NOT reproduce


## 6. Measure the pocketsA score of 1 says one way out. It does not say how much is stuck behindit. A cul-de-sac tip traps itself, which is obvious from the sidewalk. Alooping subdivision can trap dozens of blocks behind one street, which isnot. Pocket size, not score, is what makes this map worth looking at.So every trapped spot gets a minimum cut, and whatever lands on thetrapped side gets counted, and the street doing the trapping gets named.Two rules. No cutoff here: a cut-off flow cannot reliably tell you WHEREthe cut is, and the cut is the point. And only spots inside the city arecounted, because the halo is 156 km2 of ranch roads that are genuinelyremote and not interesting.

In [ ]:
import time

reds = [n for n, s in scores_h.items() if s <= 1]
print(f"measuring pockets for {len(reds):,} trapped spots\n")

pocket_size, pocket_road = {}, {}
t0 = time.time()

for i, n in enumerate(reds):
    # No cutoff. We need the actual cut, not just its size.
    cut_value, (near, far) = nx.minimum_cut(flowG_h, n, SINK)

    # Count only city spots. Halo ranch roads are true but off-topic.
    pocket_size[n] = len(near & display_nodes)

    # Name the roads doing the trapping: edges crossing from near to far.
    names = set()
    for u in near:
        for v in flowG_h[u]:
            if v in far and v != SINK:
                e = G_halo.get_edge_data(u, v)
                nm = e[0].get("name") if e else None
                if isinstance(nm, list):
                    nm = nm[0]
                if nm:
                    names.add(nm)
    pocket_road[n] = ", ".join(sorted(names)) if names else "unnamed"

    if i and i % 400 == 0:
        rate = i / (time.time() - t0)
        print(f"  {i:>5} / {len(reds)}   ~{(len(reds)-i)/rate/60:.1f} min left")

print(f"\nDone in {(time.time()-t0)/60:.1f} min 🎯\n")

# --- Distribution. Watch the first line.
sizes = sorted(pocket_size.values())
print("pocket sizes:")
print(f"  1 spot (cul-de-sac tip): {sum(1 for s in sizes if s == 1):>5}")
print(f"  2 to 5 spots           : {sum(1 for s in sizes if 2 <= s <= 5):>5}")
print(f"  6 to 20 spots          : {sum(1 for s in sizes if 6 <= s <= 20):>5}")
print(f"  21+ spots              : {sum(1 for s in sizes if s >= 21):>5}")
print(f"  biggest                : {max(sizes):>5}\n")

# --- The list that goes on the map, deduped by street name.
print("biggest traps (clipped -> halo):")
seen = set()
for n in sorted(pocket_size, key=pocket_size.get, reverse=True):
    if pocket_road[n] in seen:
        continue
    seen.add(pocket_road[n])
    d = G_halo.nodes[n]
    print(f"  {pocket_size[n]:>3} blocks behind {pocket_road[n]:<32} "
          f"({d['y']:.4f}, {d['x']:.4f})")
    if len(seen) >= 10:
        break

measuring pockets for 2,060 trapped spots

    400 / 2060   ~13.4 min left
    800 / 2060   ~10.1 min left
   1200 / 2060   ~6.9 min left
   1600 / 2060   ~3.7 min left
   2000 / 2060   ~0.5 min left

Done in 16.4 min 🎯

pocket sizes:
  1 spot (cul-de-sac tip):  1094
  2 to 5 spots           :   441
  6 to 20 spots          :   341
  21+ spots              :   184
  biggest                :    49

biggest traps (clipped -> halo):
   49 blocks behind Chatham Drive                    (38.4426, -122.7702)
   38 blocks behind Stone Bridge Road                (38.4482, -122.6132)
   25 blocks behind Northpoint Parkway               (38.4148, -122.7590)
   25 blocks behind Summerfield Road                 (38.4221, -122.6553)
   25 blocks behind Terra Linda Drive                (38.4708, -122.7169)
   22 blocks behind Village Parkway                  (38.4562, -122.6795)
   15 blocks behind San Ramon Way                    (38.4735, -122.6393)
   15 blocks behind La Mar Way                  

## 7. Export the map dataOne point per intersection, carrying whether it sits on a safety road, howmany ways out it has, how many spots share its trap, and the street doingthe trapping.Numbers, not colours. The web map decides what red means, so this filenever has to know.

In [ ]:
import json

features = []
for n in display_nodes:
    d = G_halo.nodes[n]
    lon, lat = round(d["x"], 5), round(d["y"], 5)

    if n in safe_nodes_h:
        props = {"safe": True, "score": None, "pocket": None, "road": None}
    else:
        props = {
            "safe":   False,
            "score":  scores_h.get(n),
            "pocket": pocket_size.get(n),   # None unless trapped
            "road":   pocket_road.get(n),   # the street doing it
        }

    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": props,
    })

with open("egress_santa_rosa.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)

import os
kb = os.path.getsize("egress_santa_rosa.geojson") / 1024
print(f"saved {len(features):,} points, {kb:.0f} KB ✅  (old file: 5,302 points)\n")

# --- What the map's legend will compute for itself on load.
print("what index.html will show:")
print(f"  21+ blocks trapped : {sum(1 for f in features if (f['properties']['pocket'] or 0) >= 21):>5}   (was 280)")
print(f"  6 to 20 trapped    : {sum(1 for f in features if 6 <= (f['properties']['pocket'] or 0) <= 20):>5}   (was 313)")
print(f"  dead end street    : {sum(1 for f in features if 1 <= (f['properties']['pocket'] or 0) <= 5):>5}   (was 1,348)")
print(f"  two ways out       : {sum(1 for f in features if f['properties']['score'] == 2):>5}   (was 1,355)")
print(f"  three or more      : {sum(1 for f in features if f['properties']['score'] == 3):>5}   (was 1,196)")
print(f"  safety road        : {sum(1 for f in features if f['properties']['safe']):>5}   (was 806)\n")

# --- Bridgewood autopsy. No computing, just reading what we already know.
bw = [n for n in pocket_size if "Bridgewood" in (pocket_road.get(n) or "")]
print("Bridgewood Drive (was the #1 trap at 51 blocks):")
if not bw:
    print("  no spot in Santa Rosa is trapped behind it any more. 💀")
else:
    print(f"  still trapping {max(pocket_size[n] for n in bw)} blocks, "
          f"across {len(bw)} spots")

# --- Sanity: nothing should be missing its numbers.
broken = [f for f in features if not f["properties"]["safe"]
          and f["properties"]["score"] is None]
print(f"\npoints missing a score (want 0): {len(broken)}")

saved 5,718 points, 934 KB ✅  (old file: 5,302 points)

what index.html will show:
  21+ blocks trapped :   184   (was 280)
  6 to 20 trapped    :   341   (was 313)
  dead end street    :  1535   (was 1,348)
  two ways out       :  1475   (was 1,355)
  three or more      :  1371   (was 1,196)
  safety road        :   812   (was 806)

Bridgewood Drive (was the #1 trap at 51 blocks):
  still trapping 3 blocks, across 3 spots

points missing a score (want 0): 0


## 8. Download the fileThen upload it to the repo root, where `index.html` fetches it by name.

In [ ]:
from google.colab import files
files.download("egress_santa_rosa.geojson")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>